# 03 — Performance

**Approach (course guideline):** identify the main bottleneck from measurements, apply a proportionate improvement,
and say whether further tuning is justified. The step durations of every run are recorded in the layer reports;
the comparisons below measure two design choices of the pipeline on one 10-day window (~15 M events on prod).
Code: `opdi_pipeline/performance.py`. Run it on **prod** (the dev sample has a single partition, so partition
pruning has nothing to skip there).

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))      # projet_opdi/src
import opdi_pipeline                                     # first: on the VM, findspark points to Spark 3.5 (/opt/spark)
from opdi_pipeline.config import load_settings
from opdi_pipeline.spark import get_spark

ENV = "prod"                     # "prod": full scope (~/opdi-data) | "dev": 10-day sample (~/opdi-data-dev)
settings = load_settings(ENV)
spark = get_spark(settings)
print(f"{ENV}: scope [{settings.start}, {settings.end}[ | data {settings.data_root} | Spark {spark.version}")

## 1. Where does the time go? (step durations of the last run)

In [ ]:
runs = sorted((settings.data_root / "reports").glob("*/summary.json"), key=lambda p: p.stat().st_mtime)
summary = json.loads(runs[-1].read_text())
durations = sorted(summary["step_durations_s"].items(), key=lambda kv: kv[1], reverse=True)
total = sum(d for _, d in durations)
for step, seconds in durations:
    print(f"{step:45s} {seconds:8.1f} s  {seconds / total:6.1%}")

## 2. Optimisations of the pipeline

Optimisations used in the pipeline, and measured evidence for the main ones:
- **Broadcast joins** for the small airport dimension (no shuffle of the flights): checked in the physical plan.
- **Cache** of the flagged flights (3.3 M rows, used 5 times) instead of re-reading Bronze for each use.
- **No `count()` on the 86 M events before the write**: quality figures are computed from the written tables.
- **`dropDuplicates` instead of a `row_number()` window** for the events: hash aggregation, no global sort.
- **Partition pruning**: Silver events are partitioned by `_source_period`, so a query on one 10-day window reads one
  partition only.

The measurements below are run on the last 10-day window of the scope (~15 M events) to keep them short.
In the dev run (a single 10-day window, so a single partition) partition pruning has nothing to skip: its
gain only shows on the full scope.

In [ ]:
from opdi_pipeline import performance
print("Joins of the Silver flight enrichment (expected: BroadcastHashJoin):")
for line in performance.enrichment_joins(spark, settings):
    print("  ", line[:140])

In [ ]:
print("Events x valid flights join, one 10-day window:")
print(performance.join_strategies(spark, settings))

In [ ]:
print("Partition pruning on silver/flight_events:")
print(performance.partition_pruning(spark, settings))

In [ ]:
print(performance.spark_configuration(spark))

## 3. Interpretation

*To be written from the prod measurements: main bottleneck, effect of the improvements, and why further tuning is
(or is not) justified at this data volume.*